### Importando as Bibliotecas
***

In [18]:
import pandas as pd

### Base de Dados
***

https://www.kaggle.com/datasets/saurabh00007/diabetescsv

### Ralizando a leitura dos dados
***

In [19]:
dados_diabetes = pd.read_csv('./dataset/diabetes.csv')


In [20]:
# Verificando dados do dataset
dados_diabetes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


### Tratando e analisando os dados
***

In [21]:
dados_diabetes.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [22]:
dados_diabetes.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [23]:
# Verificando valores nulos
dados_diabetes.isna().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

### Contar números de registros de cada classe
*****


In [26]:
dados_diabetes.groupby(['Outcome'])['Outcome'].count()

Outcome
0    500
1    268
Name: Outcome, dtype: int64

Verificamos que o nosso dataset se encontra desbalanceado. Precisamos coletar dados de forma balanceada afim de não prejudicar a classificação.

## Instalando biblioteca imblearn

Imblearn é uma biblioteca utilizada para tratar dados desbalanceados. A biblioteca oferece várias técnicas de reamostragem comumente usadas em conjuntos de dados que mostram um forte desequilíbrio entre as classes.

In [30]:
!pip install imblearn

In [33]:
from imblearn.under_sampling import RandomUnderSampler, TomekLinks

### Separando conjunto de dados de features e target
##### X_dados
##### y_dados
***

In [36]:
X_dados = dados_diabetes.drop('Outcome', axis=1).values
y_dados = dados_diabetes['Outcome'].values

## Criando função para balancear os dados

UnderSampler é um método consiste em reduzir o número de observações da classe majoritária para diminuir a diferença entre as categorias.
***

In [37]:
def balanceamento_dados(X_dados, y_dados):
    undersample = RandomUnderSampler(random_state=42)
    X_under, y_under = undersample.fit_resample(X_dados, y_dados)
    
    tl = TomekLinks(sampling_strategy='all')
    X_under, y_under = tl.fit_resample(X_under, y_under)
    return X_under, y_under

## Aplicando técnica de balanceamento na base de dados
***

In [41]:
X_dados_balanceados, y_dados_balanceados = balanceamento_dados(X_dados, y_dados)

## Verificando balanceamento realizado
***

In [44]:
print(len(X_dados_balanceados), len(y_dados_balanceados))

456 456


In [45]:
# verificar lista com dados balanceados
import collections
print(collections.Counter(y_dados_balanceados))

Counter({np.int64(0): 228, np.int64(1): 228})


## Criando função para realizar separação entre o conjunto de treinamento e teste

Para que o modelo de árvore de decisão tenha êxito, é necessário separarmos nosso dataset em um conjunto de dados para treinar o modelo e um conjunto de dados para testá-lo.

In [46]:
from sklearn.model_selection import train_test_split

In [48]:
def separa_treino_teste(X_dados_balanceados, y_dados_balanceados):
    X_train, X_teste, y_train, y_teste = train_test_split(X_dados_balanceados, y_dados_balanceados,
                                                          random_state= 42,
                                                          test_size=0.2
                                                         )
    return X_train, X_teste, y_train, y_teste

In [49]:
# Separando conjunts de treino e teste
X_train, X_teste, y_train, y_teste = separa_treino_teste(X_dados_balanceados, y_dados_balanceados)